In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 公开 Baseline 十二视图连续入口

一个 GPU 运行时顺序处理四片公开 Baseline；CAL32/TEST500、四配置、十二视图。不能与 public 分片或完整入口同时运行；可与两个 main 分片并行。最终用 CPU 汇总入口收集全表。

[运行顺序、分片、恢复和时长说明](https://github.com/RICHAAARC/SC-SSTW/blob/main/docs/paper_execution_v1.md)


In [ ]:
SOURCE_REF = 'main'
SOURCE_DIRECTORY = None  # 可选可读源码，不要求版本摘要一致
PREVIOUS_RUNTIME_STOPPED = False  # 前一个占用本片的 runtime 已实际停止且换机时才填 True
MAX_ITEMS = None  # 默认持续运行，不按源/pack数量自动暂停
MAX_HOURS = None  # 默认持续运行，不设置人为时长暂停；原科学资源上限保留


中断后保持 ROLE/SHARD、目录和方法不变，再次 Run all。先实际停止旧 worker；完整阶段直接复用，未完成阶段从保存输入重做。主方法与同 Wan 每源共享最多两次基础设施恢复，额外调用与耗时单列；原算法失败和旧终态失败不重试。公开 Baseline 沿用原恢复预算。

这是阶段/任务级续跑，不承诺生成步、求解迭代或 VAE 内部任意位置无损恢复。最终汇总只有一个写入者。


In [ ]:
from pathlib import Path
import urllib.request
bootstrap_path = 'runtime/wan/paper_suite_colab_v1.py'
if SOURCE_DIRECTORY:
    bootstrap = (Path(SOURCE_DIRECTORY) / bootstrap_path).read_text()
else:
    bootstrap = urllib.request.urlopen(f'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/{SOURCE_REF}/{bootstrap_path}', timeout=120).read().decode('utf-8')
entry = {'__name__': 'paper_suite_colab_entry'}
exec(compile(bootstrap, bootstrap_path, 'exec'), entry)
RUN = entry['run_public_compact_colab'](source_directory=SOURCE_DIRECTORY, source_ref=SOURCE_REF, previous_runtime_stopped=PREVIOUS_RUNTIME_STOPPED, max_items=MAX_ITEMS, max_hours=MAX_HOURS)


正式结果与失败包均保留在 `MyDrive/Video-WM_Result`；可恢复大数组放独立 `MyDrive/Video-WM_Cache`，暂停时不删除。只有完整结束的自有临时缓存可以释放。分片仅改变调度，不改变 DCT2＋CDC32、旧 H/L、固定源/视图/分母、阈值或科学调用预算。

入口返回、测量完成与科学结论分开；随机查询条件界不等于固定身份的总体 FPR 认证。公开自然视频与同 Wan 对照分表报告。


In [ ]:
print('入口状态:', RUN['status'])
print('本次记录:', RUN['invocation'])
print('各片返回:', [(row['stage'], row['result'].get('status')) for row in RUN['stages']])
entry['show_results']()
